Run this SQL once in the target PostgreSQL database before loading documents. The vector column is sized for the embeddings produced by `gemini-embedding-001`; the cosine-distance index supports nearest-neighbor searches used below.

```sql
CREATE TABLE documents_w_ownership (
    id SERIAL PRIMARY KEY,
    content TEXT NOT NULL,
    embedding VECTOR(3072),  -- 3072 dimensions for default gemini-embedding-001
    created_at TIMESTAMP DEFAULT NOW(),
    owner_id uuid,
    tenant_id uuid
);

CREATE TABLE document_acl (
    document_id BIGINT NOT NULL REFERENCES documents_w_ownership(id) ON DELETE CASCADE,
    user_id UUID NOT NULL,
    permission TEXT NOT NULL
        CHECK (permission IN ('read', 'write', 'admin')),
    PRIMARY KEY (document_id, user_id)
);

CREATE INDEX document_acl_user_permission_idx
    ON document_acl (user_id, permission, document_id);

CREATE INDEX documents_tenant_owner_idx
    ON documents_w_ownership (tenant_id, owner_id);
```
The table name must match `COLLECTION_NAME` in the following Python cells.

In [6]:
import os
from dotenv import load_dotenv
load_dotenv("/Users/admin/Documents/rag/RAG_Using_Langchain/.env")

import psycopg
from pgvector.psycopg import register_vector
from langchain_google_genai import  GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

from langchain_community.document_loaders import PyPDFLoader



/var/folders/zj/40rtcw8n3sj_y0w4jg7g9v2m0000gn/T/ipykernel_47932/1797801061.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
# Initiate the model and embeddings with the API key from environment variables
API_KEY = os.getenv("GEMINI_API_KEY")

model = "gemini-3.6-flash"

# Create the model and embeddings instances
gemini_model = ChatGoogleGenerativeAI(model=model, google_api_key=API_KEY, temperature=0.8)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", api_key=API_KEY)


In [7]:
# Initialize the loader with the path to your PDF file
pdfloader = PyPDFLoader("./whatisai.pdf")

# Load the PDF and split it into pages
pdf_pages = pdfloader.load()

In [ ]:

DATABASE_URL = os.getenv("DATABASE_URL")
COLLECTION_NAME = "documents_w_ownership" # Ensure this matches the table name in your PostgreSQL database

def load_document_to_pgvector(pdf_pages):
    with psycopg.connect(DATABASE_URL) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            # Insert documents and their embeddings into the table
            for doc in pdf_pages:
                embedding_vector = embeddings.embed_query(doc.page_content)
                try:
                    cur.execute(f"""
                        INSERT INTO {COLLECTION_NAME} (content, embedding)
                        VALUES (%s, %s)
                        """, (doc.page_content, embedding_vector))
                except Exception as exc:
                    print(f"Failed to insert document: {exc}")
            conn.commit()




In [9]:
load_document_to_pgvector(pdf_pages)   

In [ ]:
def load_acl_for_user (user_id: int, document_id: int, permission_level: str):
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            try:
                cur.execute(f"""
                    INSERT INTO document_acl (user_id, document_id, permission)
                    VALUES (%s, %s, %s)
                    """, (user_id, document_id, permission_level))
                conn.commit()
            except Exception as exc:
                print(f"Failed to insert ACL entry: {exc}")